In [1]:
import dataclasses
import matplotlib.pyplot as plt
import numpy as np
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)
from trot.prop.hubbard_cpmc_ops import _build_prop_ctx
from trot.core.system import System
from trot.ham.hubbard import HamHubbard
from trot.ham.chol import HamChol
from trot.trial.uhf import UhfTrial, get_rdm1 as uhf_get_rdm1, overlap_u as uhf_overlap_u
from trot.trial.auto import make_auto_trial_ops
from trot.core.ops import MeasOps
from trot.meas.uhf import energy_kernel_uw_rh, build_meas_ctx as uhf_build_meas_ctx
from trot.prop import blocks, cpmc_slow
from trot.prop.types import QmcParams
from trot.core.ops import k_energy
from trot.driver import run_qmc_energy
from trot.walkers import _qr
from trot import walkers as wk
from trot.prop.cpmc import init_prop_state
from trot.prop.hubbard_cpmc_ops import make_hubbard_cpmc_ops
from trot.prop.types import PropOps, PropState

In [2]:
#Hubbard model parameters
L = 8 
n_up = 4
n_down = 4 
t = 1.0
U = 4.0

h1 = np.zeros((L, L))
for i in range(L - 1):
    h1[i, i + 1] = h1[i + 1, i] = -t

# restricted HF 
eps, mo = np.linalg.eigh(h1)
Ca = mo[:, :n_up].copy()
Cb = mo[:, :n_down].copy()

e_hf = 2.0 * eps[:n_up].sum() + U * sum((Ca[i] @ Ca[i]) * (Cb[i] @ Cb[i]) for i in range(L))

ham = HamHubbard(h1=jnp.asarray(h1), u=U)
sys_ = System(norb=L, nelec=(n_up, n_down), walker_kind="unrestricted")
# the same determinant as a UHF trial: used for the energy estimator and the reference run
trial_data = UhfTrial(mo_coeff_a=jnp.asarray(Ca), mo_coeff_b=jnp.asarray(Cb))


chol = np.zeros((L, L, L))
for i in range(L):
    chol[i, i, i] = np.sqrt(U)
ham_chol = HamChol(h0=jnp.asarray(0.0), h1=jnp.asarray(h1),
                   chol=jnp.asarray(chol), basis="restricted")
meas_ctx_uhf = uhf_build_meas_ctx(ham_chol, trial_data)


def energy_uhf(walker, ham_data=None, meas_ctx=None, trial_data=trial_data):
    """Hubbard local energy via meas/uhf's Cholesky kernel. ham_data/meas_ctx are
    ignored: the propagation carries HamHubbard, this needs the HamChol form."""
    return energy_kernel_uw_rh(walker, ham_chol, meas_ctx_uhf, trial_data)
print(f"E_HF = {e_hf:.10f}")

E_HF = -1.5175409663


In [3]:
#Full ED to benchmark. Est runtime 20s

from itertools import combinations

def _hop_matrix(n):
    """Spinless nearest-neighbour hopping in the n-electron occupation basis.

    The Jordan-Wigner string between ADJACENT sites is empty, so every nonzero
    element is just -t and there is no sign to track.
    """
    strs = [frozenset(c) for c in combinations(range(L), n)]
    idx = {s: k for k, s in enumerate(strs)}
    M = np.zeros((len(strs), len(strs)))
    for k, s in enumerate(strs):
        for i in range(L - 1):
            for src, dst in ((i + 1, i), (i, i + 1)):
                if src in s and dst not in s:
                    M[idx[frozenset((s - {src}) | {dst})], k] += -t
    return M, strs


def ed_energy(u):
    """Ground state by dense eigh in the (n_up, n_down) sector."""
    Ha, sa = _hop_matrix(n_up)
    Hb, sb = _hop_matrix(n_down)
    na, nb = len(sa), len(sb)

    H = np.zeros((na * nb, na * nb))
    Hr = H.reshape(na, nb, na, nb)          # a view, so these write into H
    for ib in range(nb):
        Hr[:, ib, :, ib] += Ha              # hopping, spin up
    for ia in range(na):
        Hr[ia, :, ia, :] += Hb              # hopping, spin down
    docc = np.array([[len(a & b) for b in sb] for a in sa]).ravel()
    H[np.diag_indices_from(H)] += u * docc  # U * sum_i n_iup n_idn

    return float(np.linalg.eigvalsh(H)[0]), H.shape[0]


e_exact, ed_dim = ed_energy(U)

print(f"E_exact = {e_exact:.12f}")


E_exact = -4.235806999130


In [4]:
"""Two functions. The first grows B adapatevely until the tolarance is reached. 
The second one always use the maximum available B to avoid any possible error
and it can be used to benchmark/debug. The "plan" is (occ,B,v_ref). Fixing occ and B
keeps the shapes of the subsquent operation fixed and the whole process efficient. v_ref
is used as initial guess of the eigenvector, to avoid calling eig in the replay function. It
is not necessary for now.
"""
def plan_channel(C, eps_occ=1e-10):
    """Fishman-White compression of one spin channel. NumPy, run once.

    Records only the discrete decisions: the product-state occupations `occ`, the
    block size `B` used at each step, and the reference eigenvector `vref` that
    selects the mode (and, later, fixes its sign).
    """
    U, n = np.asarray(C, float).copy(), C.shape[0]
    occ, Bs, vrefs = np.zeros(n, int), [], []

    for k in range(n - 1):
        Lam = U @ U.T
        for B in range(2, n - k + 1):                      # grow the block until a mode isolates
            w, W = np.linalg.eigh(Lam[k : k + B, k : k + B])
            if min(w[0], 1.0 - w[-1]) < eps_occ:
                break
        if w[0] <= 1.0 - w[-1]:                            # empty mode is the cleaner one
            v, occ[k] = W[:, 0], 0
        else:                                              # occupied mode
            v, occ[k] = W[:, -1], 1
        Bs.append(B)
        vrefs.append(v.copy())

        for j in range(B - 1, 0, -1):                      # Givens-rotate v onto site k
            th = np.arctan2(v[j], v[j - 1])
            c, s = np.cos(th), np.sin(th)
            v[j - 1], v[j] = c * v[j - 1] + s * v[j], 0.0
            U[k + j - 1], U[k + j] = (
                c * U[k + j - 1] + s * U[k + j],
                -s * U[k + j - 1] + c * U[k + j],
            )

    occ[n - 1] = int(round(float(U[n - 1] @ U[n - 1])))
    assert occ.sum() == C.shape[1], "particle number lost during compression"
    return occ, np.array(Bs), vrefs

def plan_channel_maxB(C):
    """plan_channel with B = n-k always. The remaining block is then a genuine
    projector, so its eigenvalues are exactly 0/1 for any walker."""
    U, n = np.asarray(C, float).copy(), C.shape[0]
    occ, Bs, vrefs = np.zeros(n, int), [], []

    for k in range(n - 1):
        B = n - k                                          
        w, W = np.linalg.eigh((U @ U.T)[k : k + B, k : k + B])
        if w[0] <= 1.0 - w[-1]:
            v, occ[k] = W[:, 0], 0
        else:
            v, occ[k] = W[:, -1], 1
        Bs.append(B)
        vrefs.append(v.copy())
        for j in range(B - 1, 0, -1):
            th = np.arctan2(v[j], v[j - 1])
            c, s = np.cos(th), np.sin(th)
            v[j - 1], v[j] = c * v[j - 1] + s * v[j], 0.0
            U[k + j - 1], U[k + j] = (c * U[k + j - 1] + s * U[k + j],
                                      -s * U[k + j - 1] + c * U[k + j])

    occ[n - 1] = int(round(float(U[n - 1] @ U[n - 1])))
    assert occ.sum() == C.shape[1], "particle number lost during compression"
    return occ, np.array(Bs), vrefs

In [5]:
"""Jax native function that import the all dimensions estabilished in the previous cell. We fix occ,B,and vref. Fixing occ
means fixing the svd shape foe later. B are fixed for convenience, but we are actually always use max B for now. vref is not really necessary,
it is the starting guess for the squared operator, could be deleted.
"""
def channel_angles(C, plan, n_pow=25):
    
    """Replay: recompute the Givens angles from C with the plan frozen. Pure JAX.

    Also returns the rotated U. It is built here anyway, and det(U[occ, :]) is the
    gauge sign that the MPS construction drops -- so handing it back is free,
    whereas recomputing it later means a second pass over every rotation.
    """
    
    occ, Bs, vrefs = plan
    U, angles = jnp.asarray(C), []

    for k, (B, vref) in enumerate(zip(Bs, vrefs)):
        B = int(B)
        M = (U @ U.T)[k : k + B, k : k + B]
        # Dividing by the trace puts eig in [0, 1] and the empty mode
        # at 1, dominant for any C, orthonormal or not.
        M = M if occ[k] == 1 else jnp.eye(B) - M / jnp.trace(M)
        """Repeated squaring rather than eig, eig can cause problem with differentiability.
        Not an issue for now, just a suggestion from Claude. For maximum B, M is an exact projector,
        so this operation is exact (even if unneccesary)
        """
        for _ in range(n_pow):                            # M**(2**n_pow), renormalised
            M = M @ M
            M = M / jnp.linalg.norm(M)

        v = M @ jnp.asarray(vref)
        v = v / jnp.linalg.norm(v)
        v = v * jnp.sign(v @ jnp.asarray(vref))           # inherit the plan's sign

        for j in range(B - 1, 0, -1):
            th = jnp.arctan2(v[j], v[j - 1])
            c, s = jnp.cos(th), jnp.sin(th)
            v = v.at[j - 1].set(c * v[j - 1] + s * v[j]).at[j].set(0.0)
            U = U.at[k + j - 1].set(c * U[k + j - 1] + s * U[k + j]).at[k + j].set(
                -s * U[k + j - 1] + c * U[k + j]
            )
            angles.append((k + j - 1, th))


    return angles, U


In [ ]:
#Defining all functions that will be used for overlap computation
def V_hat(th):
    """Two-site number-conserving gate for one spin channel, as a (2,2,2,2) tensor."""
    c, s = jnp.cos(th), jnp.sin(th)
    g = jnp.eye(4).at[1, 1].set(c).at[1, 2].set(s).at[2, 1].set(-s).at[2, 2].set(c)
    return g.reshape(2, 2, 2, 2)


def split_full(T, ql, qr):
    
    """SVD-split a two-site tensor, keeping full rank inside each particle-number sector.
    `ql` / `qr` are the particle numbers on the outer bonds; they alone fix every
    shape here, which is what makes this safe under vmap They are fixed becuase we keep fixed the occ.
    """
    Dl, _, _, Dr = T.shape
    #Gruping indices
    M = T.reshape(Dl * 2, 2 * Dr)
    #Possible right and left charges
    rc = (ql[:, None] + np.arange(2)[None, :]).ravel()     
    cc = (qr[None, :] - np.arange(2)[:, None]).ravel()     
    A, B, qm = [], [], []
    #Iterate over all possible allowed charges
    for nm in sorted(set(rc.tolist()) & set(cc.tolist())):
        #Only allowed rows and columns
        r, c = np.where(rc == nm)[0], np.where(cc == nm)[0]
        #We can do SVD on a sub-block of M
        u, sv, vt = jnp.linalg.svd(M[np.ix_(r, c)], full_matrices=False)
        #We keep the maximum rank a.k.a. we don't discard any singular values, including 0, to keep the same shapes
        k = min(len(r), len(c))
        #The newly created tensors from the SVD
        A.append(jnp.zeros((Dl * 2, k), M.dtype).at[r].set(u[:, :k]))
        B.append(jnp.zeros((k, 2 * Dr), M.dtype).at[:, c].set(sv[:k, None] * vt[:k]))
        #Each of the new k bond states has the same quantum number qm
        qm += [nm] * k

    return (
        jnp.concatenate(A, 1).reshape(Dl, 2, -1),
        jnp.concatenate(B, 0).reshape(-1, 2, Dr),
        np.array(qm, int),
    )


def channel_mps(C, plan):
    """
    It starts with a prodcut state given by occ.  Then it gets the angles of the given rotations
    and start applyng them to the product state. After each gate we need to split the new tensor with an 
    svd. It returns the new tensorsm their quantum number and the sign convention for the new mps
    """
    occ = plan[0]
    ts = [jax.nn.one_hot(int(o), 2).reshape(1, 2, 1) for o in occ]   # product state
    qn = [np.zeros(1, int)]
    for o in occ:
        qn.append(qn[-1] + int(o))

    angles, U = channel_angles(C, plan)
    for p, th in reversed(angles):
        T = jnp.einsum("xypq,apqb->axyb", V_hat(th), jnp.tensordot(ts[p], ts[p + 1], 1))
        ts[p], ts[p + 1], qn[p + 1] = split_full(T, qn[p], qn[p + 2])

    return ts, qn, jnp.linalg.det(U[np.where(occ == 1)[0], :])


def combine(Aa, qna, Ab, qnb):
    """Interleave two d=2 channels into one d=4 MPS, local index = n_alpha + 2*n_beta."""
    ts, qn = [], [np.zeros((1, 2), int)]

    for i in range(len(Aa)):
        Dal, _, Dar = Aa[i].shape
        Dbl, _, Dbr = Ab[i].shape
        out = jnp.zeros((Dal, Dbl, 4, Dar, Dbr))
        for na in (0, 1):
            sgn = (-1.0) ** (na * qnb[i])                 # the Jordan-Wigner sign
            for nb in (0, 1):
                out = out.at[:, :, na + 2 * nb, :, :].set(
                    jnp.einsum("ar,b,bs->abrs", Aa[i][:, na, :], sgn, Ab[i][:, nb, :])
                )
        ts.append(out.reshape(Dal * Dbl, 4, Dar * Dbr))
        qn.append(np.stack([np.repeat(qna[i + 1], Dbr), np.tile(qnb[i + 1], Dar)], 1))

    return ts, qn


def sd_to_mps_gauged(ca, cb, plan_a, plan_b):
    """Slater determinant -> (d=4 MPS, gauge sign). Requires ORTHONORMAL columns."""
    ta, qna, sa = channel_mps(ca, plan_a)
    tb, qnb, sb = channel_mps(cb, plan_b)
    return combine(ta, qna, tb, qnb)[0], sa * sb


def sd_to_mps(ca, cb, plan_a, plan_b):
    """Tensors only, for the places where the gauge cancels anyway."""
    return sd_to_mps_gauged(ca, cb, plan_a, plan_b)[0]


def mps_overlap(bra, ket):
    """<bra|ket> for two MPS given as lists of (Dl, d, Dr) arrays."""
    e = jnp.ones((bra[0].shape[0], ket[0].shape[0]))
    for a, b in zip(bra, ket):
        e = jnp.tensordot(a, jnp.tensordot(e, b, ([1], [0])), ([0, 1], [0, 1]))
    return e.reshape(())

In [7]:
"""Creating the mps trial starting from thr HF orbitals with 
the two fucntions to highlight the differences. In practice we will use the max channel not
to have any approximamtion error and to compare exactly with the full HF-AFQMC run up to numerical precision"""
plan_a = plan_channel_maxB(Ca)
plan_b  = plan_channel_maxB(Cb)
plan_a_min  = plan_channel(Ca)
plan_b_min =  plan_channel(Cb)

trial = sd_to_mps(jnp.asarray(Ca), jnp.asarray(Cb), plan_a, plan_b)
trial_min = sd_to_mps(jnp.asarray(Ca), jnp.asarray(Cb), plan_a_min, plan_b_min)

print("occ_alpha     =", plan_a[0])
print(f"maximal B     : blocks {plan_a[1]}  gates {int((plan_a[1]-1).sum()):3d}"
      f"  chi {max(t.shape[0] for t in trial)}")
print(f"minimal B     : blocks {plan_a_min[1]}  gates "
      f"{int((plan_a_min[1]-1).sum()):3d}  chi {max(t.shape[0] for t in trial_min)}")
print("bond dims     =", [t.shape[0] for t in trial] + [trial[-1].shape[-1]])
print(f"<psi_T|psi_T> = {float(mps_overlap(trial, trial)):.12f}")

occ_alpha     = [1 1 1 0 0 0 0 1]
maximal B     : blocks [8 7 6 5 4 3 2]  gates  28  chi 256
minimal B     : blocks [5 4 4 3 3 2 2]  gates  16  chi 256
bond dims     = [1, 4, 16, 64, 256, 64, 16, 4, 1]
<psi_T|psi_T> = 1.000000000000


In [8]:
"""
Defining the overlap function and the MPO. Checking that MPO gives the right energy.
We are also fixing the sign convention. We have that:
|MPS> = G|occ>, so it is Givens rotations acting on the product state of occupied and unoccupied modes
G|SD> = |U>, the original Slater Determinant after having applied the Givens roattions to it 
<MPS|SD> = <occ|G|SD>=<occ|U>=det(U[occ,:])

"""

trial_sign = sd_to_mps_gauged(jnp.asarray(Ca), jnp.asarray(Cb), plan_a, plan_b)[1]

def overlap_mps(walker, trial_data=None):
    """<psi_T|SD(C)>. This is the function CPMC needs.

    <psi_T|SD(C)> = <g_T MPS_T | g_C MPS_C> = g_T g_C <MPS_T|MPS_C>, with each
    g = det(U_rot[occ, :]) handed back by the conversion.
    """
    ca, cb = walker
    walker_mps, sign_walker = sd_to_mps_gauged(ca, cb, plan_a, plan_b)
    return sign_walker * trial_sign * mps_overlap(trial, walker_mps)


def hubbard_mpo(L, t, U):
    """Dw=6 MPO. Bond basis: 0 = nothing started, 1..4 = a hop is pending, 5 = done."""
    W = np.zeros((L, 6, 4, 4, 6))
    for i in range(L):
        W[i, 0, :, :, 0] = I4
        W[i, 5, :, :, 5] = I4
        W[i, 0, :, :, 5] = U * (n_a @ n_b)                       # on-site interaction
        if i < L - 1:                                            # open a hop here
            W[i, 0, :, :, 1] = cr_a @ P_b                        # alpha: string on THIS site
            W[i, 0, :, :, 2] = an_a @ P_b
            W[i, 0, :, :, 3] = P_a @ cr_b                        # beta:  string also here
            W[i, 0, :, :, 4] = P_a @ an_b
        if i > 0:                                                # close one here
            W[i, 1, :, :, 5] = -t * an_a
            W[i, 2, :, :, 5] = -t * cr_a
            W[i, 3, :, :, 5] = -t * an_b
            W[i, 4, :, :, 5] = -t * cr_b
    return W


def apply_mpo(W, ts):
    """(W psi)[i] has bond dimension Dw*chi; the boundary MPO bonds are projected out."""
    out = []
    for i, (w, A) in enumerate(zip(W, ts)):
        A = np.asarray(A)
        if i == 0:
            w = w[0:1]
        if i == len(ts) - 1:
            w = w[:, :, :, 5:6]
        T = np.einsum("apqb,cqd->acpbd", w, A)                   # (Dw_l, Dl, 4, Dw_r, Dr)
        dl, cl, p, dr, cr = T.shape
        out.append(T.reshape(dl * cl, p, dr * cr))
    return out


def compress(ts, tol=1e-13):
    """Left-to-right QR to canonicalise, then right-to-left SVD. Exact at this tol."""
    ts = [np.asarray(t) for t in ts]
    for i in range(len(ts) - 1):
        Dl, d, Dr = ts[i].shape
        q, r = np.linalg.qr(ts[i].reshape(Dl * d, Dr))
        ts[i] = q.reshape(Dl, d, -1)
        ts[i + 1] = np.tensordot(r, ts[i + 1], axes=([1], [0]))
    for i in range(len(ts) - 1, 0, -1):
        Dl, d, Dr = ts[i].shape
        u, sv, vt = np.linalg.svd(ts[i].reshape(Dl, d * Dr), full_matrices=False)
        k = max(int((sv > tol * max(sv[0], 1e-300)).sum()), 1)
        ts[i] = vt[:k].reshape(k, d, Dr)
        ts[i - 1] = np.tensordot(ts[i - 1], u[:, :k] * sv[:k], axes=([2], [0]))
    return ts

def energy_mps(walker, ham_data=None, meas_ctx=None, trial_data=None):
    """E_loc = <H psi_T|phi> / <psi_T|phi>.

    Both contractions share the bra, and the gauge / det(R) prefactors cancel in
    the ratio -- so this needs no sign correction, unlike the overlap.
    """
    ca, cb = walker
    qa, _ = jnp.linalg.qr(ca)
    qb, _ = jnp.linalg.qr(cb)
    mps_walker = sd_to_mps(qa, qb, plan_a, plan_b)
    return mps_overlap(Hket,mps_walker) / mps_overlap(trial,mps_walker)

#Checking that MPO reproduces the right energy
# local operators, basis l = n_alpha + 2 n_beta :  |0>, |a>, |b>, |ab> = a+_a a+_b |0>
I4 = np.eye(4)
cr_a = np.zeros((4, 4)); cr_a[1, 0] = 1.0; cr_a[3, 2] = 1.0     # |0>->|a>, |b>->|ab>
cr_b = np.zeros((4, 4)); cr_b[2, 0] = 1.0; cr_b[3, 1] = -1.0    # |0>->|b>, |a>->-|ab>
an_a, an_b = cr_a.T.copy(), cr_b.T.copy()
n_a, n_b = np.diag([0., 1., 0., 1.]), np.diag([0., 0., 1., 1.])
P_a, P_b = np.diag([1., -1., 1., -1.]), np.diag([1., 1., -1., -1.])   # (-1)^n_sigma
raw = apply_mpo(hubbard_mpo(L, t, U), trial)
Hket = [jnp.asarray(t) for t in compress(raw)]

print("H|psi_T> before compression:", [t.shape[0] for t in raw] + [raw[-1].shape[-1]])
print("H|psi_T> after  compression:", [t.shape[0] for t in Hket] + [Hket[-1].shape[-1]])
print(f"\n<psi_T|H|psi_T> = {float(mps_overlap(Hket, trial) / mps_overlap(trial, trial)):.10f}")
print(f"E_HF            = {e_hf:.10f} ")

H|psi_T> before compression: [1, 24, 96, 384, 1536, 384, 96, 24, 1]
H|psi_T> after  compression: [1, 4, 16, 64, 256, 64, 16, 4, 1]

<psi_T|H|psi_T> = -1.5175409663
E_HF            = -1.5175409663 


In [ ]:
#Running the full MPS AFQMC (est runtime 40/50 s)
def run(overlap_fn, energy_fn=None):
    """Everything is held fixed except the overlap and (optionally) the energy kernel."""
    trial_ops = make_auto_trial_ops(sys_, overlap_u=overlap_fn, get_rdm1=uhf_get_rdm1)
    meas_ops = MeasOps(overlap=overlap_fn,
                       kernels={k_energy: energy_fn if energy_fn is not None else energy_uhf})
    return run_qmc_energy(
        sys=sys_,
        params=params,
        ham_data=ham,
        trial_data=trial_data,
        meas_ops=meas_ops,
        trial_ops=trial_ops,
        prop_ops=prop_ops,
        block_fn=blocks.block,
    )

params = QmcParams(
    dt=0.01,
    n_walkers=100,
    n_prop_steps=10,
    n_blocks=100,
    n_eql_blocks=50,
    weight_floor=1e-8,
    seed=1234,
)

def init_prop_state_pinned(**kwargs):
    """trot's initializer, with a strongly typed node counter.

    trot.prop.cpmc.init_prop_state sets node_encounters = jnp.asarray(0), which is
    WEAKLY typed, while every propagation step hands it back strongly typed
    (int + jnp.sum(bool)). The aval of the state therefore differs between the
    first and the second call of the jitted run_blocks in trot.driver, so the
    whole block scan is traced and compiled TWICE -- which is the two ~19 s
    equilibration blocks at the start of every run. Pinning the dtype keeps the
    aval fixed and costs one compile instead of two. Nothing under trot/ is
    modified.
    """
    state = init_prop_state(**kwargs)
    return state._replace(node_encounters=jnp.zeros((), dtype=int))


prop_ops = dataclasses.replace(
    cpmc_slow.make_prop_ops(ham, sys_.walker_kind),
    init_prop_state=init_prop_state_pinned,
)

mean_mps, err_mps, be_mps, bw_mps = run(overlap_mps, energy_mps)



Equilibration:

        block           E_blk             W        nodes      t[s]
[eql    0/50]   -1.5175409663  1.000000e+02           0       0.0
[eql   10/50]   -3.6930215731  1.144572e+02           0     157.3
[eql   20/50]   -4.2988328956  1.090102e+02           0     280.4
[eql   30/50]   -4.2025070668  1.026028e+02           0     403.2
[eql   40/50]   -4.2334378458  1.010712e+02           0     525.2
[eql   50/50]   -4.3527758910  1.010242e+02           0     647.4

Sampling:

        block           E_avg       E_err         E_block             W         nodes    dt[s/bl]     t[s]
[blk   10/100]   -4.2800242881   2.711e-02     -4.2800242881  9.991677e+01           0     77.048     770.5
[blk   20/100]   -4.2552037623   1.255e-02     -4.2303082243  9.961571e+01           0     12.378     894.3
[blk   30/100]   -4.2259876290   1.421e-02     -4.1673654046  9.944296e+01           0     12.440    1018.7
[blk   40/100]   -4.2167602876   1.439e-02     -4.1891845035  1.000424e+02   

In [ ]:
#Runining the full SD AFQMC and comparinf to previous run
mean_ref, err_ref, be_ref, bw_ref = run(uhf_overlap_u)
be_mps, be_ref = np.asarray(be_mps),np.asarray(be_ref)
print(f"E_HF                            {e_hf:.10f}")
print(f"E_exact (diagonalisation)       {e_exact:.10f}")
print(f"CPMC, all-MPS (overlap+energy)  {float(mean_mps):.10f} +- {float(err_mps):.10f}")
print(f"CPMC, all-determinant           {float(mean_ref):.10f} +- {float(err_ref):.10f}")

d = np.abs(be_mps - be_ref)
print(f"\nmean energy difference:   {abs(float(mean_mps) - float(mean_ref)):.3e}")
print(f"|E_mps - E_det| per block: max {d.max():.3e}   median {np.median(d):.3e}"
      f"   final {d[-1]:.3e}")
print(f"\n Errors per block")
print("  " + "  ".join(f"{x:.0e}" for x in d))



Equilibration:

        block           E_blk             W        nodes      t[s]
[eql    0/5]   -1.5175409663  1.000000e+01           0       0.0
[eql    1/5]   -2.6449293745  1.040552e+01           0       0.4
[eql    2/5]   -3.3123579242  1.104557e+01           0       0.4
[eql    3/5]   -3.4751975730  1.147157e+01           0       0.4
[eql    4/5]   -3.3839276460  1.161285e+01           0       0.4
[eql    5/5]   -3.8783857713  1.163254e+01           0       0.4

Sampling:

        block           E_avg       E_err         E_block             W         nodes    dt[s/bl]     t[s]
[blk    1/10]   -3.6496095241                 -3.6496095241  1.180616e+01           0      0.448       0.4
[blk    2/10]   -3.7466098296                 -3.8456564258  1.156225e+01           0      0.004       0.5
[blk    3/10]   -3.8776700560                 -4.1438016818  1.150810e+01           0      0.022       0.5
[blk    4/10]   -3.9970182348                 -4.3560322561  1.159411e+01           0 

In [11]:
"""
Here we compute a full MPS AFQMC, but using minimal B rather than maximal to evaluate how the error 
grows as the walkers get propagated. The size of the sublocks is fixed at the beginning and kept constant during 
the evolution of the walkers. Only differnece is that minimal B seems to require to orthonormalize the walkers first,
I am still trying to understand why. Est runtime 30s
"""

# the same overlap, built on the MINIMAL-B plan
trial_sign_min = sd_to_mps_gauged(jnp.asarray(Ca), jnp.asarray(Cb), plan_a_min, plan_b_min)[1]

def overlap_mps_min(walker, trial_data=None):
    """Minimal B DOES need the walker orthonormalised first -- unlike maximal B.

    With B maximal the selected mode is an exact eigenvector of the whole
    remaining block, so the rotation diagonalises Lambda exactly and the orbital
    space is preserved for any C. With B minimal the block eigenvector is only an
    approximate eigenvector of the full Lambda, and that approximation rests on
    the spectrum being {0, 1} -- which is exactly what orthonormality provides.
    Feed it a raw walker and the error is ~100%, not ~sqrt(delta).
    """
    ca, cb = walker
    qa, det_a = _qr(ca)
    qb, det_b = _qr(cb)
    walker_mps, sign_walker = sd_to_mps_gauged(qa, qb, plan_a_min, plan_b_min)
    return det_a * det_b * sign_walker * trial_sign_min * mps_overlap(trial_min, walker_mps)


f_min = jax.jit(jax.vmap(overlap_mps_min))
f_max = jax.jit(jax.vmap(overlap_mps))
f_ex = jax.jit(jax.vmap(lambda w: uhf_overlap_u(w, trial_data)))
mean_min, err_min, be_min, bw_min = run(overlap_mps_min)
be_min = np.asarray(be_min)

print(f"E_exact (diagonalisation)      {e_exact:.12f}")
print(f"CPMC, minimal-B MPS overlap    {float(mean_min):.12f} +- {float(err_min):.2e}")
print(f"CPMC, determinant overlap      {float(mean_ref):.12f} +- {float(err_ref):.2e}")
print(f"mean-energy difference         {abs(float(mean_min) - float(mean_ref)):.3e}")

d = np.abs(be_min - np.asarray(be_ref))
print(f"\n|E_minB - E_det| per block: max {d.max():.3e}   median {np.median(d):.3e} final {d[-1]:.3e}")
print(f"\n Errors per block")
print("  " + "  ".join(f"{x:.0e}" for x in d))




Equilibration:

        block           E_blk             W        nodes      t[s]
[eql    0/5]   -1.5175409663  1.000000e+01           0       0.0
[eql    1/5]   -2.6449293745  1.040552e+01           0      16.7
[eql    2/5]   -3.3123579242  1.104557e+01           0      18.0
[eql    3/5]   -3.4751975730  1.147157e+01           0      19.2
[eql    4/5]   -3.3839276460  1.161285e+01           0      20.5
[eql    5/5]   -3.8783857713  1.163254e+01           0      21.8

Sampling:

        block           E_avg       E_err         E_block             W         nodes    dt[s/bl]     t[s]
[blk    1/10]   -3.6496095512                 -3.6496095512  1.180617e+01           0     23.138      23.1
[blk    2/10]   -3.7466098423                 -3.8456564258  1.156225e+01           0      1.289      24.4
[blk    3/10]   -3.8776700623                 -4.1438016792  1.150810e+01           0      1.329      25.8
[blk    4/10]   -3.9970183769                 -4.3560327388  1.159411e+01           0 

In [ ]:
"""
Defining and testing fast sweep, computing a one site contraction and keeping left and right environment fixed.
Est runtime 20s
"""

def right_envs(walker):
    """R[x] = the contraction of sites x..L-1 of <bra|ket>. One backward pass.
    R[0][0, 0] is the full overlap, so the pre-loop overlap comes out for free.
    So basically we iterate over all sites and save the results of the contraction up
    to each local site.
    """
    R = [jnp.ones((1, 1))]
    for A, B in zip(reversed(trial),reversed(walker)):
        R.append(jnp.einsum("axc,bxd,cd->ab", A, B, R[-1]))
    return R[::-1]


def fast_sweep(ca, cb, rns, hs, w_floor):
    """The whole site loop for one walker, off a single conversion.

    Returns (ca, cb, overlap before the loop, overlap after, weight factor, nodes).

    The naive cost would be two O(chi^3) contractions per site -- one for M, one
    to push the left environment through -- but both need the same intermediate
        T[l,c,d] = Lenv[a,b] bra[a,l,c] ket[b,l,d]
    so it is built once and reused:
        M[l]        = T[l,c,d] R[c,d]     (O(4 chi^2))
        Lenv'[c,d]  = D[l] T[l,c,d]       (O(4 chi^2))
    leaving one O(chi^3) contraction per site instead of two.

    Note the walker C is updated row by row as the fields are chosen, but the MPS
    is NOT rebuilt: the sites already passed are folded into Lenv, and the sites
    still ahead are untouched, so R stays valid throughout the sweep.
    """
    walker_mps,walker_sign = sd_to_mps_gauged(ca, cb, plan_a, plan_b)
    pref = walker_sign * trial_sign
    R = right_envs(walker_mps)
    ov_in = pref * R[0][0, 0]
    Lenv = jnp.ones((1, 1))
    ov, logw = ov_in, jnp.zeros(())
    nodes = jnp.zeros((), jnp.int32)
    # the two field choices as diagonal one-site operators, l = n_a + 2 n_b
    D0 = jnp.array([1.0, hs[0, 0], hs[0, 1], hs[0, 0] * hs[0, 1]])
    D1 = jnp.array([1.0, hs[1, 0], hs[1, 1], hs[1, 0] * hs[1, 1]])

    for x in range(L):
        T = jnp.einsum("ab,alc,bld->lcd", Lenv, trial[x],walker_mps[x])    # the only O(chi^3)
        M = jnp.einsum("lcd,cd->l", T, R[x + 1])
        ov0, ov1 = pref * (D0 @ M), pref * (D1 @ M)
        r0 = jnp.where(0.5 * (ov0 / ov) < w_floor, 0.0, 0.5 * (ov0 / ov))
        r1 = jnp.where(0.5 * (ov1 / ov) < w_floor, 0.0, 0.5 * (ov1 / ov))
        nodes = nodes + (r0 <= 0.0) + (r1 <= 0.0)       # the constrained-path test
        norm = r0 + r1 + 1.0e-13
        take0 = rns[x] < r0 / norm
        D = jnp.where(take0, D0, D1)
        ov = jnp.where(take0, ov0, ov1)
        logw = logw + jnp.log(norm)
        ca = ca.at[x, :].mul(jnp.where(take0, hs[0, 0], hs[1, 0]))
        cb = cb.at[x, :].mul(jnp.where(take0, hs[0, 1], hs[1, 1]))
        Lenv = jnp.einsum("l,lcd->cd", D, T)                       # reuses T
    return ca, cb, ov_in, ov, jnp.exp(logw), nodes


def make_fast_prop_ops(ham_data, walker_kind):
    """PropOps using the sweep instead of 2L reconversions.

    Same arithmetic as trot.prop.cpmc_slow.cpmc_step -- same RNG stream, same
    weight clamps, same population control -- so the two routes are comparable
    step for step, not merely statistically. Nothing under trot/ is modified.
    """
    cpmc_ops = make_hubbard_cpmc_ops(ham_data, walker_kind)

    def step(state, *, params, ham_data, trial_data, trial_ops,
             meas_ops, meas_ctx, prop_ctx):
        key, subkey = jax.random.split(state.rng_key)
        nw = wk.n_walkers(state.walkers)
        rns = jax.random.uniform(subkey, (nw, cpmc_ops.n_sites()))
        w_floor = float(getattr(params, "weight_floor", 1.0e-8))
        w_cap = float(getattr(params, "weight_cap", 100.0))
        damping = float(getattr(params, "pop_control_damping", 0.1))

        # --- first one-body half step, then the whole site loop on one conversion ---
        walkers = cpmc_ops.apply_one_body_half(state.walkers, prop_ctx)
        ca, cb, ov_half, overlaps, wfac, nod = jax.vmap(
            fast_sweep, in_axes=(0, 0, 0, None, None))(
            walkers[0], walkers[1], rns, prop_ctx.hs_constant, w_floor)

        ratio = jnp.real(jnp.real(ov_half) / state.overlaps)
        ratio = jnp.where(ratio < w_floor, 0.0, ratio)
        nodes = jnp.sum(ratio <= 0.0) + jnp.sum(nod)
        weights = jnp.where(state.weights * ratio > w_cap, 0.0, state.weights * ratio)
        weights = weights * wfac
        walkers = (ca, cb)

        # --- second one-body half step: a general rotation, so a full overlap ---
        walkers = cpmc_ops.apply_one_body_half(walkers, prop_ctx)
        overlaps_new = jnp.real(
            jax.vmap(meas_ops.overlap, in_axes=(0, None))(walkers, trial_data))
        ratio = jnp.real(overlaps_new / overlaps)
        ratio = jnp.where(ratio < w_floor, 0.0, ratio)
        nodes = nodes + jnp.sum(ratio <= 0.0)
        weights = jnp.where(weights * ratio > w_cap, 0.0, weights * ratio)

        weights = weights * jnp.exp(prop_ctx.dt * state.pop_control_ene_shift)
        weights = jnp.where(weights > w_cap, 0.0, weights)
        avg_w = jnp.clip(jnp.mean(weights), min=1.0e-300)
        return PropState(
            walkers=walkers, weights=weights, overlaps=overlaps_new, rng_key=key,
            pop_control_ene_shift=state.e_estimate
            - damping * (jnp.log(avg_w) / prop_ctx.dt),
            e_estimate=state.e_estimate,
            node_encounters=state.node_encounters + nodes,
        )

    return PropOps(init_prop_state=init_prop_state_pinned,
                   build_prop_ctx=lambda h, t, p: _build_prop_ctx(h, p.dt),
                   step=step)

prop_ops_slow = prop_ops
prop_ops_fast = make_fast_prop_ops(ham, sys_.walker_kind)

prop_ops = prop_ops_fast
mean_fast, err_fast, be_fast, bw_fast = run(overlap_mps, energy_mps)
prop_ops = prop_ops_slow

be_fast = np.asarray(be_fast)

print(f"E_exact (diagonalisation)        {e_exact:.12f}")
print(f"CPMC, MPS + fast sweep           {float(mean_fast):.12f} +- {float(err_fast):.2e}")
print(f"CPMC, MPS + reconversion         {float(mean_mps):.12f} +- {float(err_mps):.2e}")
print(f"CPMC, all-determinant            {float(mean_ref):.12f} +- {float(err_ref):.2e}")

print(f"\nmean difference, fast vs reconversion  {abs(float(mean_fast) - float(mean_mps)):.3e}")
print(f"mean difference, fast vs determinant   {abs(float(mean_fast) - float(mean_ref)):.3e}")

d = np.abs(be_fast - be_mps)
print(f"\n|E_fast - E_reconv| per block: max {d.max():.3e}   median {np.median(d):.3e}")
print("  " + "  ".join(f"{x:.0e}" for x in d))


Equilibration:

        block           E_blk             W        nodes      t[s]
[eql    0/5]   -1.5175409663  1.000000e+01           0       0.0
[eql    1/5]   -2.6449293745  1.040552e+01           0      16.5
[eql    2/5]   -3.3123579242  1.104557e+01           0      16.8
[eql    3/5]   -3.4751975730  1.147157e+01           0      17.0
[eql    4/5]   -3.3839276460  1.161285e+01           0      17.2
[eql    5/5]   -3.8783857713  1.163254e+01           0      17.5

Sampling:

        block           E_avg       E_err         E_block             W         nodes    dt[s/bl]     t[s]
[blk    1/10]   -3.6496095241                 -3.6496095241  1.180616e+01           0     17.749      17.7
[blk    2/10]   -3.7466098296                 -3.8456564258  1.156225e+01           0      0.238      18.0
[blk    3/10]   -3.8776700560                 -4.1438016818  1.150810e+01           0      0.240      18.2
[blk    4/10]   -3.9970182348                 -4.3560322561  1.159411e+01           0 

In [13]:
#Comparing perfomances of pure jax overlap with pyblock3
import time

import pyblock3.algebra.ad as ad

ad.ENABLE_JAX = True                    # must precede the ad submodule imports
from pyblock3.algebra.symmetry import SZ
from pyblock3.algebra.ad.core import SparseTensor, SubTensor
from pyblock3.algebra.ad.mps import MPS

PHYS = np.array([[li % 2, li // 2] for li in range(4)])     # local index -> (n_a, n_b)
Q = lambda na, nb: SZ(int(na) + int(nb), int(na) - int(nb), 0)


def to_pyblock3(tensors, qn):
    """Dense (Dl, 4, Dr) arrays + (n_a, n_b) bond labels -> block-sparse pyblock3 MPS.

    A block exists only where left label + physical label = right label, which is
    what makes the representation sparse.
    """
    out = []
    for i, A in enumerate(tensors):
        blocks = []
        for nl in sorted(set(map(tuple, qn[i].tolist()))):
            rows = np.where((qn[i] == np.array(nl)).all(1))[0]
            for li in range(4):
                nr = tuple(np.array(nl) + PHYS[li])
                cols = np.where((qn[i + 1] == np.array(nr)).all(1))[0]
                if len(cols) == 0:
                    continue
                data = A[np.ix_(rows, [li], cols)]
                if float(jnp.abs(data).max()) == 0.0:
                    continue
                blocks.append(SubTensor(data=data,
                                        q_labels=(Q(*nl), Q(*PHYS[li]), Q(*nr))))
        out.append(SparseTensor(blocks=blocks))
    return MPS(tensors=out)


# combine() returns the bond labels too; sd_to_mps drops them, so call it directly
_ta, _qna, _ = channel_mps(jnp.asarray(Ca), plan_a)
_tb, _qnb, _ = channel_mps(jnp.asarray(Cb), plan_b)
ket_ts, ket_qn = combine(_ta, _qna, _tb, _qnb)
ket_pb = to_pyblock3(ket_ts, ket_qn)

# a batch of perturbed, row-scaled, non-orthonormal walkers -- built here so this
# appendix stands on its own
_rng = np.random.default_rng(0)
_nw = 32
_wu = np.stack([Ca + 0.25 * _rng.standard_normal(Ca.shape) for _ in range(_nw)])
_wd = np.stack([Cb + 0.25 * _rng.standard_normal(Cb.shape) for _ in range(_nw)])
_wu[:, 3, :] *= 2.1
_wd[:, 5, :] *= 0.4
Wb = (jnp.asarray(_wu), jnp.asarray(_wd))

qa0, _ = jnp.linalg.qr(Wb[0][0])
qb0, _ = jnp.linalg.qr(Wb[1][0])
_ta, _qna, _ = channel_mps(qa0, plan_a)
_tb, _qnb, _ = channel_mps(qb0, plan_b)
bra_ts, bra_qn = combine(_ta, _qna, _tb, _qnb)
bra_pb = to_pyblock3(bra_ts, bra_qn)

ours_val = float(mps_overlap(bra_ts, ket_ts))
pb3_val = float(bra_pb.dot(ket_pb))
print(f"blocks are jax arrays : {isinstance(ket_pb.tensors[1].blocks[0].data, jnp.ndarray)}")
print(f"<bra|ket>  dense      : {ours_val:.14f}")
print(f"<bra|ket>  pyblock3   : {pb3_val:.14f}")
print(f"           rel diff   : {abs(pb3_val / ours_val - 1):.3e}")

# ---------------------------------------------------------------- how sparse is it?
dense_n = sum(int(np.prod(t.shape)) for t in bra_ts)
block_n = sum(int(np.prod(b.data.shape)) for t in bra_pb.tensors for b in t.blocks)
print(f"\nblocks per site : {[len(t.blocks) for t in bra_pb.tensors]}")
print(f"dense bond dims : {[t.shape[0] for t in bra_ts] + [bra_ts[-1].shape[-1]]}")
print(f"stored entries  : {block_n} of {dense_n} dense  ->  {100*block_n/dense_n:.1f}%")


# ------------------------------------------------- contraction cost, same inputs
def bench(f, n=20):
    f()
    t0 = time.time()
    for _ in range(n):
        f()
    return (time.time() - t0) / n * 1e3


ket_np, bra_np = [np.asarray(t) for t in ket_ts], [np.asarray(t) for t in bra_ts]


def mps_overlap_np(bra, ket):
    e = np.ones((bra[0].shape[0], ket[0].shape[0]))
    for a, b in zip(bra, ket):
        e = np.tensordot(a, np.tensordot(e, b, ([1], [0])), ([0, 1], [0, 1]))
    return float(e.reshape(()))


jit_ov = jax.jit(mps_overlap)
jit_ov(bra_ts, ket_ts).block_until_ready()

print("\ncontraction only, tensors already built, per walker:")
print(f"  dense tensordot, numpy    : {bench(lambda: mps_overlap_np(bra_np, ket_np)):8.3f} ms")
print(f"  dense tensordot, jnp      : {bench(lambda: float(mps_overlap(bra_ts, ket_ts))):8.3f} ms")
print(f"  dense tensordot, jitted   : {bench(lambda: jit_ov(bra_ts, ket_ts).block_until_ready()):8.3f} ms")
print(f"  pyblock3 .dot             : {bench(lambda: float(bra_pb.dot(ket_pb))):8.3f} ms")
print(f"\nand just building the pyblock3 MPS : {bench(lambda: to_pyblock3(bra_ts, bra_qn)):8.3f} ms")

# ------------------------------------------------- end to end over the batch
f_ours = jax.jit(jax.vmap(lambda w: mps_overlap(
    sd_to_mps(jnp.linalg.qr(w[0])[0], jnp.linalg.qr(w[1])[0], plan_a, plan_b), ket_ts)))
f_ours(Wb).block_until_ready()
t_ours = bench(lambda: f_ours(Wb).block_until_ready(), n=10)


def pb3_batch():
    out = []
    for k in range(Wb[0].shape[0]):
        qa, _ = jnp.linalg.qr(Wb[0][k])
        qb, _ = jnp.linalg.qr(Wb[1][k])
        _a, _qa, _ = channel_mps(qa, plan_a)
        _b, _qb, _ = channel_mps(qb, plan_b)
        ts, qn = combine(_a, _qa, _b, _qb)
        out.append(float(to_pyblock3(ts, qn).dot(ket_pb)))
    return np.array(out)


r_pb3 = pb3_batch()
t_pb3 = bench(pb3_batch, n=1)
nw = Wb[0].shape[0]
print(f"\nmax rel diff over {nw} walkers : {np.max(np.abs(r_pb3 / np.asarray(f_ours(Wb)) - 1)):.3e}")
print(f"end to end, {nw} walkers:")
print(f"  ours, jit + vmap          : {t_ours:9.2f} ms  ({t_ours/nw:7.3f} ms/walker)")
print(f"  pyblock3, python loop     : {t_pb3:9.2f} ms  ({t_pb3/nw:7.2f} ms/walker)"
      f"   -> {t_pb3/t_ours:.0f}x slower")

blocks are jax arrays : True
<bra|ket>  dense      : -0.35806205341943
<bra|ket>  pyblock3   : -0.35806205341943
           rel diff   : 2.220e-16

blocks per site : [4, 16, 36, 64, 64, 36, 16, 4]
dense bond dims : [1, 4, 16, 64, 256, 64, 16, 4, 1]
stored entries  : 10680 of 139808 dense  ->  7.6%

contraction only, tensors already built, per walker:
  dense tensordot, numpy    :    0.328 ms
  dense tensordot, jnp      :    1.304 ms
  dense tensordot, jitted   :    1.012 ms
  pyblock3 .dot             :   63.628 ms

and just building the pyblock3 MPS :   92.200 ms

max rel diff over 32 walkers : 3.220e-15
end to end, 32 walkers:
  ours, jit + vmap          :     24.27 ms  (  0.758 ms/walker)
  pyblock3, python loop     :  13268.23 ms  ( 414.63 ms/walker)   -> 547x slower
